# Detect extrusions (or another event) on a movie

Used already trained neural network(s) to detect extrusions or other events from movie(s).

DeXtrusion calculates a probability map which reflects the probability of having an extrusion (or other event) for each pixel at each time of the movie. The probability map can be converted to single point position as ImageJ ROI. The position of an event will be at the centroid of a high probability volume.

## Initialization

In [1]:
import os
from dextrusion.DeXtrusion import DeXtrusion
from dextrusion.DialogDeXtrusion import DialogDeXtrusion

# default parameters
talkative = True  ## print info messages
dexter = DeXtrusion(verbose=talkative)

imname = "../data/notum_mix"    ## folder where data are saved to initialize the file browser
modeldir = os.path.join(os.path.abspath(os.path.join(os.getcwd(), os.pardir)), "DeXNets", "notum_all")  ## folder where models are saved to initialize the file browser
groupsize = 150000       ## number of windows that are runned through the neural network at the same time - depends on the computer processing capabilities
                         ## higher will run faster, but too high can fail

2026-03-04 16:32:25.583230: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-04 16:32:25.693114: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/gaelle/Logiciel/mambaforge/envs/dextrusenv/lib/python3.10/site-packages/cv2/../../lib64:
2026-03-04 16:32:25.693139: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-03-04 16:32:25.721360: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin

Tensorflow with Cuda: True
Tensorflow version: 2.10.1
Num GPUs Available:  0


2026-03-04 16:32:27.226659: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/gaelle/Logiciel/mambaforge/envs/dextrusenv/lib/python3.10/site-packages/cv2/../../lib64:
2026-03-04 16:32:27.226680: W tensorflow/stream_executor/cuda/cuda_driver.cc:263] failed call to cuInit: UNKNOWN ERROR (303)
2026-03-04 16:32:27.226694: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (gailua): /proc/driver/nvidia/version does not exist
2026-03-04 16:32:27.227269: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Graphical interface pop-up to choose the detection parameters

In [2]:
# Choose parameters through graphical interface
diag = DialogDeXtrusion()
diag.dialog_main(modeldir, imname)

## typical cell diameter in the movie. Used to resize the movie if necessary to match with the reference scale used in the training.
cell_diameter = float(diag.cell_diameter)          
## typical duration of an extrusion event (number of frames it is visible). Used to resize the movie to the reference scale if necessary
extrusion_duration = float(diag.extrusion_duration) 
## xy step used to move the sliding windows
shift_xy = int(diag.dxyval)
## frame interval used to move the sliding windows
shift_t = int(diag.dtval)
## save the resulting probability maps (of all events)
save_proba = bool(diag.saveProba)
## save the cleaned (thresholded) probability map of the selected event
save_proba_one = bool(diag.saveProbaOne)
## save the ROIs file of all the events
save_rois = bool(diag.saveRois)
## Choose which event to save the probamap of. (Put None to do all events)
cat = int(diag.event_ind)
## Names of the possible events
catnames = diag.event
## min average probability in volume to keep ROI
roi_threshold = float(diag.threshold) 
## min positive volume to keep ROI
roi_volume = float(diag.proba_vol) 
## computational parameters, number of windows to evaluate at the time
groupsize = int(diag.group_size)       

modeldir = diag.modeldir
from glob import glob
# if folder "modeldir" is not one model directory but contains several models subdirectory, use them all
if not os.path.exists( os.path.join(modeldir,"config.cfg") ):
    model = glob(modeldir+"/*/", recursive = False)
else:
    model = [modeldir]
imname = diag.imagepath
if talkative:
    print("Using network(s) "+str(model))
    print("On movie(s) "+str(imname))

['/home/gaelle/Proj/RL/dextrusion/DeXNets/notum_all/notumAll0/', '/home/gaelle/Proj/RL/dextrusion/DeXNets/notum_all/notumAll1/']
/home/gaelle/Proj/RL/dextrusion/DeXNets/notum_all/notumAll0/
Using network(s) ['/home/gaelle/Proj/RL/dextrusion/DeXNets/notum_all/notumAll0/', '/home/gaelle/Proj/RL/dextrusion/DeXNets/notum_all/notumAll1/']
On movie(s) ['/home/gaelle/Proj/RL/dextrusion/data/004-crop.tif', '']


### Run the dextrusion detection on the selected movie(s) with the selected trained network(s)

In [ ]:
## Merge "peaks" if distance < disxy (spatial) & dist (time) 
disxy = 10
distime = 4
# Parameters: astime: save tif image as temporal stack (else slice stack)
temporal = True

for image in imname:
    print(image)
    if os.path.exists(image):
        if talkative:
            print( "Detecting events on movie "+str(image) )
        
        ## Detect events
        dexter.detect_events_onmovie( image, models=model, 
                                     cell_diameter=cell_diameter, extrusion_duration=extrusion_duration, 
                                     dxy=shift_xy, dz=shift_t, 
                                     group_size=groupsize )
          
        ## Save probability maps to file, to visualize the results
        if save_proba:
            dexter.write_probamaps(cat=None, astime=temporal)
        
        ## Save cleaned probability map(s) to file, keeping only events above the size and probability thresholds (for visualisation) 
        if save_proba_one:
            dexter.write_cleanedprobamap( cat=cat, volume_threshold=roi_volume, proba_threshold=roi_threshold, 
                                         disxy=disxy, distime=distime, astime=temporal )
        
        ## Look for centroid of "positive" volumes in the probability maps and convert it to a point event 
        ## Keep only ROIs that have a big enough volume ('volume_threshold') and with high enough probability ('proba_threshold') 
        # - put 0 and 0 to keep all detections
        if save_rois:
            dexter.get_rois(cat=cat, volume_threshold=roi_volume, proba_threshold=roi_threshold, disxy=disxy, dist=distime)

/home/gaelle/Proj/RL/dextrusion/data/004-crop.tif
Detecting events on movie /home/gaelle/Proj/RL/dextrusion/data/004-crop.tif
  Initial image shape: (20, 316, 372)
  Rescaled image shape: (20, 316, 372)
  Extended image shape: (30, 316, 372)
